# FOD tespiti: Colab defteri

Bu defter veriyi hazırlar, modeli eğitir, val üzerinde değerlendirir, işlemci hızını ölçer ve en sonda nihai modelleri test setinde değerlendirir.

**Nasıl çalıştırılır:**
1. Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seç.
2. Aşağıdaki ilk kod hücresinde **yalnızca `IS = ...` satırını** seç:
   - `"yeniden_degerlendir"`: önceki deneyi yeni kodla yeniden değerlendirir, **eğitim yapmaz**.
   - `"egit"`: yeni deneyi eğitir ve değerlendirir.
   - `"hiz_olc"`: modellerin yalnızca işlemciyle hızını ölçer, **eğitim yapmaz**.
   - `"test"`: nihai modelleri test setinde **bir kez** değerlendirir. Bunun için ayrıca `TEST_ONAY = "evet"` yazılmalı.
3. Menüden *Çalışma zamanı → Tümünü çalıştır*.
4. Drive bağlama hücresinde izin penceresi açılır; izin ver. Gerisi kendiliğinden ilerler.
5. Bir `KONTROL` hücresi `HATA` ya da kırmızı yazı verirse defter orada durur; çıktıyı gönder.

In [ ]:
# NE YAPILACAK? Yalnızca bu satırı değiştir, sonra menüden: Çalışma zamanı > Tümünü çalıştır
#   "yeniden_degerlendir" : REEVAL_NAME deneyini yeni kodla val'de yeniden değerlendirir. Eğitim YOK.
#   "egit"                : EXP_NAME deneyini eğitir ve val'de değerlendirir.
#   "hiz_olc"             : modellerin yalnızca işlemciyle (ekran kartı olmadan) hızını ölçer. Eğitim YOK.
#   "test"                : nihai modelleri TEST setinde değerlendirir. Her model için YALNIZCA BİR KEZ yapılır.
IS = "yeniden_degerlendir"
assert IS in ("yeniden_degerlendir", "egit", "hiz_olc", "test"), f"IS geçersiz: {IS!r}"

# Test için ikinci kilit: IS = "test" iken bu satır da "evet" olmalı, yoksa test çalışmaz.
TEST_ONAY = "hayir"

print("Seçilen iş:", IS)
if IS == "test":
    assert TEST_ONAY == "evet", 'Test seçildi ama TEST_ONAY = "evet" yazılmadı. Test çalıştırılmadı.'
    print("TEST MODU: nihai modeller test setinde birer kez değerlendirilecek.")

# Zip Drive'da başka bir yerdeyse ZIP_PATH satırını düzelt
REPO_URL = "https://github.com/ogutalanummugulsum-netizen/fod-tespit.git"  # GitHub depon
ZIP_PATH = "/content/drive/MyDrive/FODPascalVOCFormat-V.2.1.zip"  # Drive'daki veri zip'i

# Bunlara dokunmana gerek yok
PROJECT_DIR = "/content/proje1"  # kodun Colab'da duracağı yer
DRIVE_OUT = "/content/drive/MyDrive/fod_proje"  # sonuçların Drive'da saklanacağı klasör
VOC_DIR = "data/raw/FODPascalVOCFormat-V.2.1/VOC2007"

In [ ]:
# GPU açık mı?
import torch

if torch.cuda.is_available():
    print("GPU hazır:", torch.cuda.get_device_name(0))
else:
    print("GPU YOK. Veri hazırlama için sorun değil, ama eğitimden önce GPU'yu aç.")

In [ ]:
# Google Drive'ı bağla (izin penceresi açılır)
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Kodu GitHub'dan çek ve paketleri kur
import os

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Veri zip'ini Drive'dan Colab diskine aç (Drive'dan doğrudan okumak çok yavaş)
if not os.path.exists(VOC_DIR):
    os.makedirs("data/raw", exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d data/raw

!ls {VOC_DIR}

## 1. Etiketleri YOLO formatına çevir

In [ ]:
!python scripts/convert_voc_to_yolo.py --voc-dir {VOC_DIR} --out-dir data/yolo

In [ ]:
# KONTROL: dönüşüm beklenen sayıları veriyor mu?
# Beklenen sayılar, ham veri ayrı bir yöntemle sayılarak bulundu.
from pathlib import Path

import pandas as pd

boxes = pd.read_csv("data/yolo/boxes.csv", dtype={"image_id": str})
class_counts = boxes["class_name"].value_counts()

checks = [
    ("Görüntü sayısı", boxes["image_id"].nunique(), 33793),
    ("Kutu sayısı", len(boxes), 34472),
    ("Sınıf sayısı", boxes["class_name"].nunique(), 31),
    ("Bolt kutusu", int(class_counts.get("Bolt", 0)), 3300),
    ("Screw kutusu", int(class_counts.get("Screw", 0)), 157),
    ("Tape kutusu", int(class_counts.get("Tape", 0)), 127),
    ("Tek sınıflı etiket dosyası", len(list(Path("data/yolo/all/labels_fod").glob("*.txt"))), 33793),
    ("31 sınıflı etiket dosyası", len(list(Path("data/yolo/all/labels_31").glob("*.txt"))), 33793),
    ("Koordinatlar 0-1 arasında", bool(boxes[["x_center", "y_center", "width", "height"]].stack().between(0, 1).all()), True),
]

all_ok = True
for name, found, expected in checks:
    ok = found == expected
    all_ok = all_ok and ok
    print(f"{'OK  ' if ok else 'HATA'}  {name}: bulunan {found}, beklenen {expected}")

assert all_ok, "Dönüşüm beklenen sayıları vermedi. Devam etme, önce nedenini bul."
print("\nDönüşüm doğru görünüyor.")

## 2. Train / val / test bölmesi

In [ ]:
!python scripts/make_splits.py --voc-dir {VOC_DIR} --data-dir data/yolo

In [ ]:
# KONTROL: bölme kurallara uyuyor mu? (script'ten bağımsız olarak yeniden hesaplanır)
splits = pd.read_csv("configs/splits.csv")
image_splits = pd.read_csv("data/yolo/image_splits.csv", dtype={"image_id": str})
df = boxes.merge(image_splits, on="image_id")
df["kucuk"] = df[["xmax", "ymax"]].sub(df[["xmin", "ymin"]].values).max(axis=1) < 32

total = int(splits["n_images"].sum())
frames = splits.groupby("split")["n_images"].agg(["sum", "max", "count"])
# Bir sınıfı içeren kare sayısı (bölüm başına)
class_frames = df.drop_duplicates(["image_id", "class_name"]).groupby(["split", "class_name"]).size()
small_share = 100 * df.groupby("split")["kucuk"].mean()
test_classes = set(class_frames["test"].index)
train_classes = set(class_frames["train"].index)
unseen = sorted(test_classes - train_classes)
val_top = class_frames["val"] / frames.loc["val", "sum"]

checks = [("Video parçası sayısı", len(splits), 116), ("Toplam kare", total, 33793)]
for name in ["val", "test"]:
    ratio = frames.loc[name, "sum"] / total
    share = frames.loc[name, "max"] / frames.loc[name, "sum"]
    diff = small_share[name] - small_share["train"]
    checks.append((f"{name} oranı %9-%11 arasında (bulunan %{100 * ratio:.1f})", bool(0.09 <= ratio <= 0.11), True))
    checks.append((f"{name} içinde en büyük parça yarıdan az (bulunan %{100 * share:.1f})", bool(share <= 0.5), True))
    checks.append((f"{name}: 32 pikselden küçük kutu oranı train'e yakın (fark {diff:+.1f} puan)", bool(abs(diff) <= 5), True))
for name in ["Bolt", "Nut", "Rock"]:
    checks.append((f"test'te {name} var", name in test_classes, True))
checks.append((f"val'de en az 12 parça (bulunan {frames.loc['val', 'count']})", bool(frames.loc["val", "count"] >= 12), True))
checks.append((f"val'de tek sınıf en fazla %25 (en büyük: {val_top.idxmax()} %{100 * val_top.max():.1f})", bool(val_top.max() <= 0.25), True))
checks.append((f"test'te eğitimde olmayan 2-4 sınıf (bulunan: {', '.join(unseen) or '-'})", 2 <= len(unseen) <= 4, True))

all_ok = True
for name, found, expected in checks:
    ok = found == expected
    all_ok = all_ok and ok
    print(f"{'OK  ' if ok else 'HATA'}  {name}: bulunan {found}, beklenen {expected}")

assert all_ok, "Bölme kurallara uymuyor. Devam etme."
print("\nKüçük kutu oranı (%):")
print(small_share.round(1))
splits[splits["split"] != "train"]

In [ ]:
# Sızıntı kontrolü (birkaç dakika sürebilir; bölme donduğu için hız ölçümünde ve test'te atlanır)
if IS in ("yeniden_degerlendir", "egit"):
    !python scripts/check_splits.py --data-dir data/yolo
else:
    print(f"Sızıntı kontrolü atlandı (IS = {IS!r}).")

## 3. Kutu boyutu raporu
Cisimler görüntünün ne kadarını kaplıyor? Gerçek pist kamerasında cisimler çok küçük görünür; bu tablo veri setinin buna ne kadar benzediğini gösterir.

In [ ]:
image_splits = pd.read_csv("data/yolo/image_splits.csv", dtype={"image_id": str})
df = boxes.merge(image_splits, on="image_id")

# Kutunun görüntü alanına oranı (%) ve uzun kenarı (piksel)
df["alan_yuzde"] = 100 * df["width"] * df["height"]
df["uzun_kenar_px"] = df[["width", "height"]].max(axis=1) * df["img_w"]

print("Kutunun görüntü alanına oranı (%):")
print(df.groupby("split")["alan_yuzde"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2))

df["boyut"] = pd.cut(
    df["uzun_kenar_px"],
    bins=[0, 16, 32, 64, 96, 10000],
    labels=["<16 px", "16-32 px", "32-64 px", "64-96 px", ">=96 px"],
    right=False,
)
size_table = (pd.crosstab(df["boyut"], df["split"], normalize="columns") * 100).round(1)
print("\nKutunun uzun kenarına göre dağılım (her bölümdeki kutuların yüzdesi):")
print(size_table)

os.makedirs("results", exist_ok=True)
size_table.to_csv("results/box_size_report.csv")

## 4. Çıktıları Drive'a kaydet
Colab oturumu kapanınca `/content` içindeki her şey silinir. Bölme kaydı ve raporlar Drive'a kopyalanır.

In [ ]:
import shutil

os.makedirs(f"{DRIVE_OUT}/configs", exist_ok=True)
shutil.copy("configs/splits.csv", f"{DRIVE_OUT}/configs/splits.csv")
# Yalnızca veri hazırlama raporları kopyalanır; Drive'daki deney sonuçlarına dokunulmaz
os.makedirs(f"{DRIVE_OUT}/results", exist_ok=True)
for report in ["split_report.txt", "split_check.txt", "box_size_report.csv"]:
    shutil.copy(f"results/{report}", f"{DRIVE_OUT}/results/{report}")
print("Kaydedildi:", DRIVE_OUT)

## 5. Eğitim

Aşağıdaki ayar hücresi hangi deneyin çalışacağını belirler. Şu anki deney small modelin asıl eğitimidir (`exp003_small`): nano ile aynı ayarlar, en fazla 50 epoch, val puanı 10 epoch boyunca iyileşmezse erken durur. Small, nano'dan yavaştır; epoch süresini ilk epoch bitince eğitim çıktısında görürsün. (Nano'da bir epoch yaklaşık 2,5 dakika sürmüştü.)

**Ara kayıtlar nerede?** Eğitim her epoch sonunda modeli Google Drive'a yazar:
`Drive'ım / fod_proje / runs / <deney adı> / weights / last.pt`
Colab oturumu kapansa da bu dosya silinmez.

**Oturum koparsa ne yapacağım?**
1. Defteri yeniden aç ve GPU'yu seç.
2. En baştaki hücreleri yeniden çalıştır: ayarlar, GPU kontrolü, Drive'ı bağla, kodu çek, zip'i aç.
3. `convert_voc_to_yolo.py` ve `make_splits.py` hücrelerini yeniden çalıştır (veri Colab diskinde durur, oturumla birlikte silinir). Sızıntı kontrolü ve kutu boyutu hücrelerini atlayabilirsin.
4. Aşağıdaki ayar hücresini **aynı `EXP_NAME` ile** ve eğitim hücresini yeniden çalıştır.

Eğitim hücresi Drive'daki `last.pt` dosyasını kendisi bulur ve kaldığı epoch'tan devam eder. Başka bir şey yazmana gerek yok. Eğitim zaten bitmişse yeniden eğitmez, bunu söyler.

**Dikkat:** yeni bir deney için `EXP_NAME` değerini değiştir. Aynı adı farklı ayarla kullanırsan eski deney devam eder, yeni ayarlar yok sayılır.

**Eğitim sürerken:** Colab sekmesini kapatma ve bilgisayarı uyku moduna alma. Sekme uzun süre boşta kalırsa Colab oturumu kesebilir.

In [ ]:
# DENEY AYARLARI
# IS = "egit" iken eğitilecek deney:
EXP_NAME = "exp003_small"        # deney adı; yeni deneyde değiştir
MODEL = "yolo11s.pt"             # yolo11n.pt = nano, yolo11s.pt = small
EPOCHS = 50                      # en fazla bu kadar epoch
PATIENCE = 10                    # val puanı 10 epoch iyileşmezse eğitim erken durur
IMGSZ = 320                      # görüntü boyutu
NOTE = "small asil egitim"       # summary.csv'ye yazılacak not

# IS = "yeniden_degerlendir" iken yeniden değerlendirilecek deney (eğitim yapılmaz):
REEVAL_NAME = "exp002_nano"
REEVAL_NOTE = "asil egitim"

# IS = "hiz_olc" iken hızı ölçülecek deneyler, ölçüm yeri ve iş parçacığı sayıları:
SPEED_MODELS = "exp002_nano exp003_small"
SPEED_WHERE = "colab_cpu"
SPEED_THREADS = "1 2"

# IS = "test" iken test setinde değerlendirilecek nihai modeller (README, "Nihai modeller"):
TEST_MODELS = "exp002_nano exp003_small"

RUNS_DIR = f"{DRIVE_OUT}/runs"        # ara kayıtlar ve model dosyaları (Drive)
RESULTS_DIR = f"{DRIVE_OUT}/results"  # deney sonuçları (Drive)

# Bu çalıştırmada değerlendirilecek deney
NAME = EXP_NAME if IS == "egit" else REEVAL_NAME
shown = {"hiz_olc": SPEED_MODELS, "test": TEST_MODELS}.get(IS, NAME)
print(f"İş: {IS}   Deney: {shown}")

In [ ]:
# EĞİTİM: yalnızca IS = "egit" iken çalışır
# (oturum koparsa aynı EXP_NAME ile Tümünü çalıştır; eğitim kaldığı yerden devam eder)
if IS == "egit":
    !python scripts/train.py --name {EXP_NAME} --model {MODEL} --epochs {EPOCHS} --patience {PATIENCE} --imgsz {IMGSZ} --runs-dir "{RUNS_DIR}" --results-dir "{RESULTS_DIR}"
else:
    print(f"Eğitim atlandı (IS = {IS!r}).")

## 6. Değerlendirme (val)

Modeli **val** seti üzerinde ölçer ve kırılım tablosunu çıkarır. Test setine dokunmaz.

- Güven eşiği README'deki kurala göre seçilir: val'in tamamında F2'yi en yüksek yapan eşik. Precision, recall ve hata sayıları bu eşikle verilir.
- Yanlış alarmlar ikiye ayrılır: **cisimde** (cismin parçası ya da ikinci kutu) ve **zeminde** (boş zeminde alarm).
- `video_parcasi` sütunu satırın kaç videoya dayandığını gösterir; 3'ten azsa satır "az veri" diye işaretlenir. Val'de her video için ayrı bir satır da vardır.

In [ ]:
# DEĞERLENDİRME (val): "egit" iken yeni eğitilen deney, "yeniden_degerlendir" iken REEVAL_NAME
if IS == "egit":
    !python scripts/evaluate.py --name {EXP_NAME} --split val --runs-dir "{RUNS_DIR}" --results-dir "{RESULTS_DIR}" --note "{NOTE}"
elif IS == "yeniden_degerlendir":
    !python scripts/evaluate.py --name {REEVAL_NAME} --split val --runs-dir "{RUNS_DIR}" --results-dir "{RESULTS_DIR}" --note "{REEVAL_NOTE}"
else:
    print(f"Değerlendirme atlandı (IS = {IS!r}).")

In [ ]:
# KONTROL: sonuç dosyaları Drive'a yazıldı mı? (bu çalıştırmada değerlendirilen deney: NAME)
import glob
import json

import pandas as pd

if IS not in ("yeniden_degerlendir", "egit"):
    print(f"Değerlendirme kontrolü atlandı (IS = {IS!r}).")
else:
    expected_files = [
        f"{RUNS_DIR}/{NAME}/weights/best.pt",
        f"{RUNS_DIR}/{NAME}/weights/last.pt",
        f"{RESULTS_DIR}/{NAME}/train_info.json",
        f"{RESULTS_DIR}/{NAME}/environment.txt",
        f"{RESULTS_DIR}/{NAME}/training_curve.csv",
        f"{RESULTS_DIR}/{NAME}/conf_threshold.json",
        f"{RESULTS_DIR}/{NAME}/val_breakdown.csv",
        f"{RESULTS_DIR}/{NAME}/val_metrics.json",
        f"{RESULTS_DIR}/{NAME}/val_thresholds.csv",
        f"{RESULTS_DIR}/{NAME}/val_errors/ACIKLAMA.txt",
        f"{RESULTS_DIR}/summary.csv",
    ]
    all_ok = True
    for path in expected_files:
        ok = os.path.exists(path)
        all_ok = all_ok and ok
        print(f"{'OK  ' if ok else 'YOK '}  {path}")
    print("Hata örneği görüntü sayısı:", len(glob.glob(f"{RESULTS_DIR}/{NAME}/val_errors/*.jpg")))
    print("test_log.csv var mı (olmamalı, test çalıştırılmadı):", os.path.exists(f"{RESULTS_DIR}/test_log.csv"))
    assert all_ok, "Bazı sonuç dosyaları eksik."

    thresholds = json.load(open(f"{RESULTS_DIR}/{NAME}/conf_threshold.json", encoding="utf-8"))
    assert "conf_f2" in thresholds, "conf_threshold.json içinde F2 eşiği yok: değerlendirme yeni kodla çalışmamış."
    print(f"\nGüven eşiği: F2 kuralıyla {thresholds['conf_f2']} (F1 eşiği {thresholds['conf_f1']:.3f})")

    info = json.load(open(f"{RESULTS_DIR}/{NAME}/train_info.json", encoding="utf-8"))
    curve = pd.read_csv(f"{RESULTS_DIR}/{NAME}/training_curve.csv")
    best_epoch = int(curve.loc[curve["metrics/mAP50-95(B)"].idxmax(), "epoch"])
    print(f"{NAME}: eğitim {len(curve)} epoch sürdü (en fazla {info['epochs']}), en iyi epoch {best_epoch}, toplam {curve['time'].iloc[-1] / 60:.0f} dakika.")

    print("\nsummary.csv:")
    print(pd.read_csv(f"{RESULTS_DIR}/summary.csv").to_string(index=False))

## 7. Hız ölçümü (yalnızca işlemci)

`IS = "hiz_olc"` iken çalışır. `SPEED_MODELS` içindeki her modelin, ekran kartı olmadan, yalnızca işlemciyle bir görüntüyü ne kadar sürede işlediğini ölçer. Val'den sabit 300 görüntü kullanılır (ilk 20 görüntü ısınma, ölçüme katılmaz); test setine dokunulmaz. Dosya boyutu ve parametre sayısı da kaydedilir. Sonuç: `results/speed_cpu.csv`.

In [ ]:
# HIZ ÖLÇÜMÜ: yalnızca IS = "hiz_olc" iken çalışır (birkaç dakika sürer)
if IS == "hiz_olc":
    !python scripts/benchmark_cpu.py --models {SPEED_MODELS} --where {SPEED_WHERE} --threads {SPEED_THREADS} --runs-dir "{RUNS_DIR}" --results-dir "{RESULTS_DIR}"

    # KONTROL: sonuç dosyası yazıldı mı, her model her iş parçacığı sayısıyla ölçüldü mü?
    import pandas as pd
    speed_file = f"{RESULTS_DIR}/speed_cpu.csv"
    assert os.path.exists(speed_file), f"Hız sonucu yazılmamış: {speed_file}"
    speed = pd.read_csv(speed_file)
    here = speed[speed["olcum_yeri"] == SPEED_WHERE]
    measured = set(zip(here["exp_id"], here["is_parcacigi"]))
    wanted = {(m, int(t)) for m in SPEED_MODELS.split() for t in SPEED_THREADS.split()}
    missing = wanted - measured
    assert not missing, f"Ölçülmeyen model / iş parçacığı: {missing}"
    print("\nOK: bütün modeller, bütün iş parçacığı sayılarıyla ölçüldü.")
    print(here[["exp_id", "is_parcacigi", "parametre_sayisi", "dosya_boyutu_mb", "uctan_uca_ortanca_ms", "uctan_uca_fps", "tarih"]].to_string(index=False))
else:
    print(f"Hız ölçümü atlandı (IS = {IS!r}).")

## 8. Test seti (nihai değerlendirme)

Yalnızca `IS = "test"` **ve** `TEST_ONAY = "evet"` iken çalışır. `TEST_MODELS` içindeki nihai modeller test setinde **birer kez**, kendi val F2 eşikleriyle değerlendirilir. Raporda neler verileceği README'de, test çalıştırılmadan önce yazıldı ("Test raporunda verilecekler").

Korumalar:
- `evaluate.py` test için `--final-test` ister.
- Her test değerlendirmesi `results/test_log.csv` dosyasına yazılır; aynı model için ikinci kez çalışmaz.
- Bir model yarıda hata verirse sonuçları ekrana basılmaz ve log'a yazılmaz; defter yeniden çalıştırılabilir. Önceden tamamlanan model ikinci kez değerlendirilmez.

## Sonuçları göndermek
Drive'da `fod_proje` klasörünü aç, içindeki **`results`** klasörüne sağ tıkla → **İndir**. İnen zip İndirilenler klasöründe kalsın. (`runs` klasöründe model dosyaları var; onlar Drive'da kalır, indirmene gerek yok.)

In [ ]:
# TEST: yalnızca IS = "test" ve TEST_ONAY = "evet" iken çalışır
if IS == "test":
    assert TEST_ONAY == "evet", 'TEST_ONAY = "evet" değil. Test çalıştırılmadı.'
    for test_name in TEST_MODELS.split():
        print(f"\n========== {test_name}: test değerlendirmesi ==========")
        !python scripts/evaluate.py --name {test_name} --split test --final-test --runs-dir "{RUNS_DIR}" --results-dir "{RESULTS_DIR}" --note "nihai test"

    # KONTROL: iki model de test'te tamamlandı mı?
    import pandas as pd
    log_file = f"{RESULTS_DIR}/test_log.csv"
    assert os.path.exists(log_file), "test_log.csv yazılmamış: test değerlendirmesi tamamlanmadı. Çıktıyı gönder."
    log = pd.read_csv(log_file)
    missing = set(TEST_MODELS.split()) - set(log["exp_id"])
    assert not missing, f"Test'i tamamlanmayan model: {missing}. Çıktıyı gönder; defter yeniden çalıştırılabilir."
    for test_name in TEST_MODELS.split():
        path = f"{RESULTS_DIR}/{test_name}/test_breakdown.csv"
        assert os.path.exists(path), f"Test kırılım tablosu yok: {path}"
    print("\nOK: bütün nihai modeller test setinde birer kez değerlendirildi.")
    print(log.to_string(index=False))
else:
    print(f"Test atlandı (IS = {IS!r}).")